# Collision dataset validation

## tl;dr
This notebook reruns independent full-table validation of the prepared collision dataset. Read the executed totals and checks below before using the table. No exploratory relationships, predictive models, transforms, or fitted features are produced.

## Context & Methods
The unit is one candidate collision identified by selected annual source, collision year, and case. DuckDB checks the complete Parquet against the previous acquisition audit; archived source SHA256 fingerprints are recomputed. A bounded 100-row sample per source uses an independent reader to verify canonical collision values.

### Key assumptions
Original selected NCDB releases and the completed audit are the structural baseline. Collision report conditions have not been verified as prospectively available. Undocumented predictor values are retained and exposed; missing person identities do not remove target-valid events. Validation establishes structural preparation integrity, not causal interpretation, representativeness, or prospective model performance.


## Data
Run from the project root or its notebooks folder using the project `.venv` interpreter. The required inputs are `data/sources.json`, `data/derived/quality_audit.json`, `data/reference/modeling_contract.json`, original selected raw files, and the prepared Parquet/manifest. Run `python -m road_to_safety.prepare_collisions --workers 2` first if outputs are missing.

In [1]:
from pathlib import Path
import json
import pandas as pd
from road_to_safety.validate_prepared import run

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'data/sources.json').exists():
    ROOT = ROOT / 'projects/road_to_safety'
assert (ROOT / 'data/sources.json').exists(), 'Use the project root as the working directory.'
TABLES = ROOT / 'reports/tables'


### Recompute independent checks
The validator raises an exception if any structural check fails. It saves a JSON receipt and inspectable CSV evidence.

In [2]:
receipt = run(ROOT)
pd.DataFrame([{
    key: receipt[key] for key in ['status', 'collisions', 'source_rows', 'fatal_collisions',
                                  'injury_collisions', 'ineligible_collisions', 'exception_rows']
}]).T.rename(columns={0: 'Observed value'})


,Observed value
status,passed
collisions,2949846
source_rows,7729886
fatal_collisions,47793
injury_collisions,2902053
ineligible_collisions,0
exception_rows,332599


## Results
### Annual row and target conservation
Each selected source must match the previous audit. Counts are source records or candidate collisions as labeled; they are not people at risk or a denominator for accident likelihood.

In [3]:
annual = pd.read_csv(TABLES / 'validation_annual.csv')
annual[['C_YEAR', 'collisions', 'source_rows', 'fatal_collisions', 'injury_collisions',
        'ineligible_collisions', 'exact_duplicate_extra_rows']]


,C_YEAR,collisions,source_rows,fatal_collisions,injury_collisions,ineligible_collisions,exact_duplicate_extra_rows
0,1999,151315,413509,2632,148683,0,0
1,2000,155838,422075,2548,153290,0,0
2,2001,151438,409389,2415,149023,0,0
3,2002,156415,420008,2583,153832,0,0
4,2003,152980,407036,2487,150493,0,0
5,2004,147588,389050,2438,145150,0,0
6,2005,148111,386470,2551,145560,0,0
7,2006,145103,378523,2586,142517,0,0
8,2007,141070,368507,2455,138615,0,0
9,2008,129764,338268,2193,127571,0,0


### Audit reconciliation
Every independently recomputed annual metric must match the prior full-source audit.

In [4]:
reconciliation = pd.read_csv(TABLES / 'validation_audit_reconciliation.csv')
reconciliation.groupby('metric', as_index=False).agg(
    source_year_checks=('matches', 'size'), all_match=('matches', 'all'))


,metric,source_year_checks,all_match
0,collisions,23,True
1,dummy_person_rows,23,True
2,duplicate_identified_person_keys,23,True
3,exact_duplicate_extra_rows,23,True
4,fatal_collisions,23,True
5,identified_person_keys,23,True
6,injury_collisions,23,True
7,pedestrian_dummy_vehicle_rows,23,True
8,person_severity_conflict_keys,23,True
9,unresolved_person_identity_rows,23,True


### Collision-level missing-code states
Every feature uses its full collision denominator in each year. Unknown, unavailable, other, unresolved, blank, and conflicting states are preserved rather than imputed. These are integrity summaries, not an analysis of severity relationships.

In [5]:
missingness = pd.read_csv(TABLES / 'validation_missingness.csv')
state_counts = missingness.pivot_table(index='field', columns='value_class',
                                      values='collisions', aggfunc='sum', fill_value=0)
assert (state_counts.sum(axis=1) == receipt['collisions']).all()
state_counts


value_class,other,stated,unknown
field,,,
C_HOUR,0,2918585,31261
C_MNTH,0,2949734,112
C_RALN,14738,2742254,192854
C_RCFG,95746,2619639,234461
C_RSUR,87977,2824246,37623
C_TRAF,42317,2772718,134811
C_WDAY,0,2949320,526
C_WTHR,6775,2897662,45409


### Dictionary domains and exception evidence
Undocumented values are listed without silently assigning dictionary meanings or excluding otherwise eligible events.

In [6]:
domain_exceptions = pd.read_csv(TABLES / 'validation_domain_exceptions.csv')
display(domain_exceptions.head(25))
pd.DataFrame.from_dict(receipt['exception_checks'], orient='index', columns=['Observed count'])


,field,C_YEAR,value,conflict,collisions,dictionary_supported


,Observed count
unique_exception_source_row,0
exception_has_reason,0
invalid_key_exception_conservation,0
unresolved_person_exception_conservation,30589
dummy_person_exception_conservation,15555
pedestrian_dummy_exception_conservation,304229
duplicate_person_exception_conservation,1794
valid_exception_collision_join,0


### Source preservation and bounded value checks
Hashes establish byte preservation relative to the acquisition manifest. Bounded value checks supplement the full-table count and domain checks; they do not constitute a second full raw-file value audit.

In [7]:
assert all(row['matches_acquisition_and_audit'] for row in receipt['raw_fingerprints'])
assert not receipt['failures']
pd.DataFrame([{
    'Raw file fingerprints verified': len(receipt['raw_fingerprints']),
    'Annual sources sampled': len(receipt['bounded_source_value_samples']),
    'Source rows compared': sum(row['source_rows_checked'] for row in receipt['bounded_source_value_samples']),
    'Output and code fingerprints recorded': len(receipt['input_fingerprints']),
    'Structural failures': len(receipt['failures']),
}])


,Raw file fingerprints verified,Annual sources sampled,Source rows compared,Output and code fingerprints recorded,Structural failures
0,24,23,2300,7,0


## Takeaways
Use `data/derived/collisions.parquet` with the versioned feature/target contract for subsequent descriptive work. Filter `severity_eligible` for future severity modeling and select only the eight approved condition features. Exception and diagnostic fields are quality evidence and are excluded predictors.

The JSON validation receipt and CSV tables retain exact counts, fingerprints, and exceptions. The saved notebook must execute without error. Visual inspection of the saved notebook has not been completed in this environment; open it in Jupyter to inspect the bounded tables. No model training, EDA, accident probability estimation, or causal policy claims are part of this phase.
